# TUGAS MANDIRI PRAKTIKUM BIG DATA — PERTEMUAN 11
## Machine Learning dengan Spark MLlib: Feature Engineering & Regresi

| Informasi | Detail |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Pertemuan** | 11 (Tugas Mandiri) |
| **Nama Mahasiswa** | Yarsya |
| **NPM** | 2505060066 |
| **Program Studi** | S1 Teknologi Informasi — Universitas Tidar |

---

### Skenario Tugas
Manajemen **"Kopi Nusantara"** membutuhkan alat bantu prediksi omzet harian tiap cabang untuk merencanakan stok bahan baku secara presisi. Tugas ini membangun model **Regresi Linear (`LinearRegression`)** di PySpark MLlib untuk memprediksi `total_penjualan` sebuah pesanan berdasarkan atribut produk, cabang, waktu, jumlah unit, dan harga satuan, lengkap dengan eksperimen perbandingan fitur serta inferensi pada skenario baru.

## 0. Inisialisasi SparkSession & Persiapan Dataset
Menyiapkan `SparkSession`, membangkitkan dataset `tugas11_penjualan.csv` (2.500 baris), dan membacanya ke dalam Spark DataFrame.

In [1]:
import os
import sys
import numpy as np
import pandas as pd

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder \
    .appName("Tugas11-MLlib-Regresi-Yarsya") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Membangkitkan dataset tugas11_penjualan.csv
np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

# Membaca dataset dengan Spark
df = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)
print(f"Dataset berhasil dimuat: {df.count()} baris")
df.printSchema()
df.show(5)

ERROR StatusLogger Could not determine local host name
 java.net.UnknownHostException: sandbox: sandbox: Temporary failure in name resolution
	at java.base/java.net.InetAddress.getLocalHost(InetAddress.java:1670)
	at org.apache.logging.log4j.core.util.NetUtils.getLocalHostname(NetUtils.java:56)
	at org.apache.logging.log4j.core.LoggerContext.lambda$setConfiguration$0(LoggerContext.java:615)
	at java.base/java.util.concurrent.ConcurrentHashMap.computeIfAbsent(ConcurrentHashMap.java:1705)
	at org.apache.logging.log4j.core.LoggerContext.setConfiguration(LoggerContext.java:615)
	at org.apache.logging.log4j.core.LoggerContext.reconfigure(LoggerContext.java:694)
	at org.apache.logging.log4j.core.LoggerContext.reconfigure(LoggerContext.java:711)
	at org.apache.logging.log4j.core.LoggerContext.start(LoggerContext.java:253)
	at org.apache.logging.log4j.core.impl.Log4jContextFactory.getContext(Log4jContextFactory.java:245)
	at org.apache.logging.log4j.core.impl.Log4jContextFactory.getContext(Log

ERROR StatusConsoleListener Could not determine local host name
 java.net.UnknownHostException: sandbox: sandbox: Temporary failure in name resolution
	at java.base/java.net.InetAddress.getLocalHost(InetAddress.java:1670)
	at org.apache.logging.log4j.core.util.NetUtils.getLocalHostname(NetUtils.java:56)
	at org.apache.logging.log4j.core.LoggerContext.lambda$setConfiguration$0(LoggerContext.java:615)
	at java.base/java.util.concurrent.ConcurrentHashMap.computeIfAbsent(ConcurrentHashMap.java:1705)
	at org.apache.logging.log4j.core.LoggerContext.setConfiguration(LoggerContext.java:615)
	at org.apache.logging.log4j.core.LoggerContext.reconfigure(LoggerContext.java:694)
	at org.apache.logging.log4j.core.LoggerContext.setConfigLocation(LoggerContext.java:679)
	at org.apache.spark.internal.Logging.initializeLogging(Logging.scala:139)
	at org.apache.spark.internal.Logging.initializeLogIfNecessary(Logging.scala:114)
	at org.apache.spark.internal.Logging.initializeLogIfNecessary$(Logging.scala:1

ERROR StatusConsoleListener Could not determine local host name
 java.net.UnknownHostException: sandbox: sandbox: Temporary failure in name resolution
	at java.base/java.net.InetAddress.getLocalHost(InetAddress.java:1670)
	at org.apache.logging.log4j.core.util.NetUtils.getLocalHostname(NetUtils.java:56)
	at org.apache.logging.log4j.core.LoggerContext.lambda$setConfiguration$0(LoggerContext.java:615)
	at java.base/java.util.concurrent.ConcurrentHashMap.computeIfAbsent(ConcurrentHashMap.java:1705)
	at org.apache.logging.log4j.core.LoggerContext.setConfiguration(LoggerContext.java:615)
	at org.apache.logging.log4j.core.LoggerContext.reconfigure(LoggerContext.java:694)
	at org.apache.logging.log4j.core.LoggerContext.reconfigure(LoggerContext.java:711)
	at org.apache.spark.internal.Logging$.uninitialize(Logging.scala:220)
	at org.apache.spark.deploy.SparkSubmit.org$apache$spark$deploy$SparkSubmit$$runMain(SparkSubmit.scala:972)
	at org.apache.spark.deploy.SparkSubmit.doRunMain$1(SparkSubmit

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/26 16:11:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/11/26 16:11:22 WARN MacAddressUtil: Failed to find a usable hardware address from the network interfaces; using random bytes: a3:f5:79:5d:90:ed:e3:d7


Dataset berhasil dimuat: 2500 baris
root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- total_penjualan: double (nullable = true)

+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows


---
## Bagian A. Feature Engineering (Bobot 20%)

Menggunakan `StringIndexer` untuk mengubah kolom kategorikal (`nama_menu`, `cabang`, `waktu_pesan`) menjadi indeks numerik, lalu menggabungkan seluruh fitur (`qty`, `harga_satuan`, dan ketiga kolom hasil indexing) menjadi satu vektor kolom `features` dengan `VectorAssembler`.

In [2]:
# Mendefinisikan StringIndexer untuk setiap kolom kategorikal
idx_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx")

# Melatih (fit) masing-masing indexer dan menyimpannya agar dapat dipakai ulang pada Bagian E
model_idx_menu = idx_menu.fit(df)
df_idx = model_idx_menu.transform(df)

model_idx_cabang = idx_cabang.fit(df_idx)
df_idx = model_idx_cabang.transform(df_idx)

model_idx_waktu = idx_waktu.fit(df_idx)
df_idx = model_idx_waktu.transform(df_idx)

# Menggabungkan seluruh fitur dengan VectorAssembler
assembler_lengkap = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCol="features"
)

df_final = assembler_lengkap.transform(df_idx)
print("Hasil Feature Engineering (5 baris pertama):")
df_final.select("nama_menu", "nama_menu_idx", "cabang", "cabang_idx", "waktu_pesan", "waktu_pesan_idx", "features", "total_penjualan").show(5, truncate=False)

Hasil Feature Engineering (5 baris pertama):


+--------------+-------------+----------+----------+-----------+---------------+-------------------------+------------------+
|nama_menu     |nama_menu_idx|cabang    |cabang_idx|waktu_pesan|waktu_pesan_idx|features                 |total_penjualan   |
+--------------+-------------+----------+----------+-----------+---------------+-------------------------+------------------+
|Kentang Goreng|5.0          |Semarang  |2.0       |Malam      |1.0            |[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|Kentang Goreng|5.0          |Yogyakarta|1.0       |Sore       |3.0            |[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|Kentang Goreng|5.0          |Semarang  |2.0       |Siang      |2.0            |[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|Kentang Goreng|5.0          |Yogyakarta|1.0       |Pagi       |0.0            |[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|Roti Bakar    |4.0          |Magelang  |0.0       |Malam      |1.0            |[2.0,30000.0,4.0,0.0,1.0]|58157.944120

---
## Bagian B. Train/Test Split & Pelatihan Model (Bobot 20%)

Membagi data dengan proporsi **75% training** dan **25% testing** (`seed=101`), kemudian melatih model `LinearRegression` dengan target `total_penjualan`.

In [3]:
# Split data 75:25 dengan seed=101
train_data, test_data = df_final.randomSplit([0.75, 0.25], seed=101)

print(f"Jumlah data training : {train_data.count()} baris")
print(f"Jumlah data testing  : {test_data.count()} baris")

# Melatih model LinearRegression
lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model_lr = lr.fit(train_data)

print("\nModel Regresi Linear berhasil dilatih!")
print("Koefisien [qty, harga_satuan, menu_idx, cabang_idx, waktu_idx]:", model_lr.coefficients)
print(f"Intercept: {model_lr.intercept:.2f}")

Jumlah data training : 1839 baris
Jumlah data testing  : 661 baris



Model Regresi Linear berhasil dilatih!
Koefisien [qty, harga_satuan, menu_idx, cabang_idx, waktu_idx]: [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept: -66903.18


---
## Bagian C. Evaluasi Model (Bobot 20%)

Mengukur akurasi prediksi model pada data testing menggunakan metrik **RMSE** dan **R²**.

In [4]:
# Menghasilkan prediksi pada data testing
predictions = model_lr.transform(test_data)

evaluator_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")

rmse_utama = evaluator_rmse.evaluate(predictions)
r2_utama = evaluator_r2.evaluate(predictions)

print("=== HASIL EVALUASI MODEL UTAMA (5 FITUR) ===")
print(f"RMSE : Rp {rmse_utama:,.2f}")
print(f"R²   : {r2_utama:.4f} ({r2_utama * 100:.2f}% variasi data dijelaskan)")

predictions.select("nama_menu", "cabang", "qty", "harga_satuan", "total_penjualan", "prediction").show(5)

=== HASIL EVALUASI MODEL UTAMA (5 FITUR) ===
RMSE : Rp 10,900.80
R²   : 0.9107 (91.07% variasi data dijelaskan)
+---------+--------+---+------------+------------------+------------------+
|nama_menu|  cabang|qty|harga_satuan|   total_penjualan|        prediction|
+---------+--------+---+------------+------------------+------------------+
|Americano|Magelang|  1|       25000|19721.170446051627|31056.146105506836|
|Americano|Magelang|  1|       25000|  21236.0956379149|31056.146105506836|
|Americano|Magelang|  4|       22000|  85834.8564401191| 88127.37331410211|
|Americano|Magelang|  4|       22000|102174.09100317833| 88127.37331410211|
|Americano|Magelang|  4|       30000|113044.63431555482|112310.74025938749|
+---------+--------+---+------------+------------------+------------------+
only showing top 5 rows



### Interpretasi Evaluasi Model (Bagian C)
Model regresi linear ini **cukup baik dan layak** digunakan untuk memprediksi omzet karena nilai **R² di atas 0,88** (mampu menjelaskan hampir 90% variasi total penjualan dari fitur masukan). Kesalahan prediksi rata-rata (**RMSE**) sekitar Rp 11 ribuan masih tergolong wajar mengingat adanya fluktuasi acak transaksi dan efek perkalian non-linear `qty * harga_satuan` yang didekati dengan garis linear.

---
## Bagian D. Eksperimen Perbandingan Fitur (Bobot 25%)

Melatih **model kedua** yang HANYA menggunakan 2 fitur numerik inti (`qty` dan `harga_satuan`) tanpa menyertakan kolom kategorikal (`nama_menu`, `cabang`, `waktu_pesan`), kemudian membandingkan metrik R² dan RMSE dengan model utama.

In [5]:
# Menyusun VectorAssembler kedua hanya dengan qty dan harga_satuan
assembler_sederhana = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_v2"
)

train_v2 = assembler_sederhana.transform(train_data)
test_v2 = assembler_sederhana.transform(test_data)

lr_v2 = LinearRegression(featuresCol="features_v2", labelCol="total_penjualan")
model_lr_v2 = lr_v2.fit(train_v2)
pred_v2 = model_lr_v2.transform(test_v2)

rmse_v2 = evaluator_rmse.evaluate(pred_v2)
r2_v2 = evaluator_r2.evaluate(pred_v2)

print("=== PERBANDINGAN MODEL UTAMA (5 FITUR) VS MODEL KEDUA (2 FITUR) ===")
print(f"Model 1 (qty, harga, menu, cabang, waktu) -> RMSE: Rp {rmse_utama:,.2f} | R²: {r2_utama:.4f}")
print(f"Model 2 (HANYA qty, harga_satuan)         -> RMSE: Rp {rmse_v2:,.2f} | R²: {r2_v2:.4f}")
print(f"Selisih R²: {abs(r2_utama - r2_v2):.6f}")

=== PERBANDINGAN MODEL UTAMA (5 FITUR) VS MODEL KEDUA (2 FITUR) ===
Model 1 (qty, harga, menu, cabang, waktu) -> RMSE: Rp 10,900.80 | R²: 0.9107
Model 2 (HANYA qty, harga_satuan)         -> RMSE: Rp 10,896.03 | R²: 0.9108
Selisih R²: 0.000078


### Kesimpulan Eksperimen Perbandingan Fitur (Bagian D)
Menambahkan informasi `nama_menu`, `cabang`, dan `waktu_pesan` **tidak meningkatkan akurasi prediksi secara berarti** (selisih R² antara Model 1 dan Model 2 sangat kecil, kurang dari 0,001). Hal ini terjadi karena secara matematis nilai `total_penjualan` ditentukan hampir sepenuhnya oleh kuantitas (`qty`) dan `harga_satuan`, sedangkan nama menu, lokasi cabang, maupun waktu pemesanan dipilih secara acak independen dalam data. Dalam praktik Data Science (*prinsip parsimony*), model kedua dengan 2 fitur justru lebih disukai karena lebih sederhana, lebih cepat dilatih, dan bebas dari *noise* fitur yang tidak relevan.

---
## Bagian E. Prediksi Skenario Baru (Bobot 15%)

Membuat DataFrame berisi **3 skenario pesanan baru (data dummy)**, memprosesnya menggunakan objek `StringIndexerModel` yang **sama persis** dari hasil `fit()` pada Bagian A beserta `assembler_lengkap`, lalu menampilkan hasil prediksi `total_penjualan` dari model regresi utama.

In [6]:
# Membuat 3 skenario pesanan baru (data dummy)
skenario_baru = [
    ("Kopi Susu", "Magelang", "Pagi", 2, 22000),
    ("Cappuccino", "Yogyakarta", "Sore", 4, 25000),
    ("Roti Bakar", "Semarang", "Malam", 1, 18000),
]
kolom_skema = ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"]
df_baru = spark.createDataFrame(skenario_baru, schema=kolom_skema)

# Menerapkan model StringIndexer yang SUDAH di-fit pada Bagian A (tidak membuat StringIndexer baru)
df_baru_idx = model_idx_menu.transform(df_baru)
df_baru_idx = model_idx_cabang.transform(df_baru_idx)
df_baru_idx = model_idx_waktu.transform(df_baru_idx)

# Menggabungkan menjadi vector features
df_baru_features = assembler_lengkap.transform(df_baru_idx)

# Menghasilkan prediksi menggunakan model_lr
hasil_prediksi_baru = model_lr.transform(df_baru_features)

print("Hasil Prediksi Total Penjualan untuk 3 Skenario Baru:")
hasil_prediksi_baru.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction").show(truncate=False)

Hasil Prediksi Total Penjualan untuk 3 Skenario Baru:


+----------+----------+-----------+---+------------+-----------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction       |
+----------+----------+-----------+---+------------+-----------------+
|Kopi Susu |Magelang  |Pagi       |2  |22000       |43571.19372074085|
|Cappuccino|Yogyakarta|Sore       |4  |25000       |97908.2695218741 |
|Roti Bakar|Semarang  |Malam      |1  |18000       |9961.412053669395|
+----------+----------+-----------+---+------------+-----------------+



In [7]:
# Menutup SparkSession
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
